# Example 3 Custom Function Wrapping Multiple Models

**Objective:** Custom pyfunc that chooses between two internal sklearn models.

This example is independent and can run directly.

In [ ]:

import os
import sys
import json
import time
import math
import pickle
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import (
    load_iris, load_diabetes, load_digits, load_linnerud,
    load_wine, load_breast_cancer, make_classification, make_regression
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")


def find_project_root(start_path=None):
    """Find the mlflow_for_ml_dev folder from wherever Jupyter is started."""
    start = Path(start_path or Path.cwd()).resolve()
    for parent in [start] + list(start.parents):
        if parent.name == "mlflow_for_ml_dev" and (parent / "notebooks").exists():
            return parent
        if (parent / "datasets").exists() and (parent / "mlruns").exists() and (parent / "notebooks").exists():
            return parent
    # Fallback: if user started Jupyter inside notebooks subfolder
    for parent in [start] + list(start.parents):
        if parent.name == "notebooks":
            return parent.parent
    return start

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "datasets"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "artifacts_generated"

for folder in [DATASETS_DIR, MLRUNS_DIR, MODELS_DIR, ARTIFACTS_DIR, MLRUNS_DIR / ".trash"]:
    folder.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Every notebook logs to the same mlruns folder inside this project.
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
client = MlflowClient()

print("Project root      :", PROJECT_ROOT)
print("Datasets folder   :", DATASETS_DIR)
print("MLflow runs folder:", MLRUNS_DIR)
print("Models folder     :", MODELS_DIR)
print("Tracking URI      :", mlflow.get_tracking_uri())


## 1. Setup

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import mlflow.pyfunc

X, y = make_classification(n_samples=500, n_features=6, random_state=42)
X = pd.DataFrame(X, columns=[f"f{i}" for i in range(6)])
log_model = LogisticRegression(max_iter=300).fit(X, y)
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)

class ModelRouter(mlflow.pyfunc.PythonModel):
    def __init__(self, linear_model, tree_model):
        self.linear_model = linear_model
        self.tree_model = tree_model
    def predict(self, context, model_input):
        use_tree = model_input.get("use_tree", pd.Series([0]*len(model_input))).astype(bool)
        features = model_input[[c for c in model_input.columns if c.startswith("f")]]
        linear_preds = self.linear_model.predict(features)
        tree_preds = self.tree_model.predict(features)
        return np.where(use_tree, tree_preds, linear_preds)

mlflow.set_experiment("independent_example_model_router")
with mlflow.start_run() as run:
    mlflow.pyfunc.log_model("model", python_model=ModelRouter(log_model, tree_model))
    uri = f"runs:/{run.info.run_id}/model"
loaded = mlflow.pyfunc.load_model(uri)
test = X.head(5).copy(); test["use_tree"] = [0,1,0,1,1]
print(loaded.predict(test))